In [1]:
import brainpy as bp
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import jax
import jax.numpy as jnp
import sys
import os

src_dir = os.path.abspath(os.path.join('../../'))
sys.path.insert(0, src_dir)
import src

/headnode2/bhar9988/code/DDC/WRCircuit.jl/.CondaPkg/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from src.positions import ClusteredPositions
from src.neurons import LIFNeuron
from src.synapses import DeltaSynapse

# Everything here runs fast enough
N=50000
epsilon=0.1
D=1.5
nu_hat=2
g=5
J=0.1

num_inh = N // 5
num_exc = N - num_inh

theta = 20  # mV
tau = 20  # ms
tau_ref = 2.0  # ms
nu_thr = (
    1000 * theta / (epsilon * num_exc * J * tau)
)  # Tau is in milliseconds. *1000 to convert to Hz
nu = nu_hat * nu_thr

# geometry
exc_positions = ClusteredPositions((-1.5, 0), 1)
inh_positions = ClusteredPositions((1.5, 0), 1)

# neurons
E = LIFNeuron(
    size=num_exc,
    embedding=exc_positions,
    V_rest=0.0,  # For simple IF neuron in paper
    V_th=theta,
    V_reset=10.0,
    R=1,
    tau=tau,
    tau_ref=tau_ref,
    V_initializer=bp.init.Normal(0, 1.0),
)

# Create a population of inhibitory neurons
I = LIFNeuron(
    size=num_inh,
    embedding=inh_positions,
    V_rest=0.0,
    V_th=theta,
    V_reset=10.0,
    R=1,
    tau=tau,
    tau_ref=tau_ref,
    V_initializer=bp.init.Normal(0, 1.0),
)

# Synapses
delay_step = D // bp.share["dt"]
delay_step = int(delay_step)

JE = J
JI = -g * J

In [75]:
conn=bp.connect.FixedProb(prob=epsilon, allow_multi_conn=True)
f = conn(10000, 10000)
f.build_csr()

(Array([1453, 4192, 2420, ..., 5195,  398, 3624], dtype=int32),
 Array([       0,     1000,     2000, ...,  9998000,  9999000, 10000000],      dtype=int32))

In [62]:
f.build_mat()

Array(value=Array([[False, False,  True, ..., False, False, False],
                   [False, False, False, ...,  True, False, False],
                   [False, False, False, ..., False, False, False],
                   ...,
                   [False, False, False, ...,  True, False, False],
                   [False, False, False, ...,  True, False, False],
                   [False, False, False, ..., False, False, False]]),
      dtype=bool)

In [ ]:
self = bp.dyn.TwoEndConn(pre=E, post=E)
self.__init__( pre=E, post=E, conn=bp.connect.FixedProb(prob=epsilon), output=bp._src.dynold.synouts.CUBA(target_var="V"), stp=None, mode=None
)
# connections and weights
self.g_max, self.conn_mask = self._init_weights(1.0, comp_method="sparse", sparse_data="csr"
)


# # register delay
# self.delay_step = delay_step
# self.pre.register_local_delay("spike", self.name, delay_step=self.delay_step)


/headnode2/bhar9988/code/DDC/WRCircuit.jl/.CondaPkg/env/lib/python3.11/site-packages/brainpy/_src/deprecations.py:89: DeprecationWarning: brainpy.dyn.TwoEndConn is deprecated. Use brainpy.synapses.TwoEndConn instead.
  _deprecate(message)
2024-12-08 16:00:24.195750: W external/tsl/tsl/framework/bfc_allocator.cc:482] Allocator (GPU_0_bfc) ran out of memory trying to allocate 5.96GiB (rounded to 6400000000)requested by op 
2024-12-08 16:00:24.195853: W external/tsl/tsl/framework/bfc_allocator.cc:494] *********************************__**____*********************______________________________________
E1208 16:00:24.195878  513090 pjrt_stream_executor_client.cc:2985] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 6400000000 bytes.
/headnode2/bhar9988/code/DDC/WRCircuit.jl/.CondaPkg/env/lib/python3.11/site-packages/brainpy/_src/connect/random_conn.py:514: DeprecationWarning: invalid escape sequence '\_'
  """Build a Watts–Strogatz small-world graph

XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 6400000000 bytes.

In [ ]:
conn_mask = self.conn.require("pre2post") # This is it

In [77]:
%load_ext line_profiler
%lprun -f DeltaSynapse.__init__ DeltaSynapse(E, E, bp.connect.FixedProb(prob=epsilon, allow_multi_conn=True), delay_step=delay_step, g_max=JE)

The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler


Timer unit: 1e-09 s

Total time: 0.494492 s
File: /headnode2/bhar9988/code/DDC/WRCircuit.jl/src/synapses.py
Function: __init__ at line 88

Line #      Hits         Time  Per Hit   % Time  Line Contents
    88                                               def __init__(
    89                                                   self,
    90                                                   pre: NeuDyn,
    91                                                   post: NeuDyn,
    92                                                   conn: Union[TwoEndConnector, ArrayType, Dict[str, ArrayType]],
    93                                                   output: SynOut = CUBA(target_var="V"),
    94                                                   stp: Optional[SynSTP] = None,
    95                                                   comp_method: str = "sparse",
    96                                                   g_max: Union[float, ArrayType, Initializer, Callable] = 1.0,
    97              

In [ ]:
# * Takes about 20 seconds. Can we jax it?
E2E = DeltaSynapse( # * This is the slow part. Can we jax it?
    E,
    E,
    bp.connect.FixedProb(prob=epsilon),
    delay_step=delay_step,
    g_max=JE,
)

0


In [ ]:


E2I = DeltaSynapse(
    E,
    I,
    bp.connect.FixedProb(prob=epsilon),
    delay_step=delay_step,
    g_max=JE,
)
I2E = DeltaSynapse(
    I,
    E,
    bp.connect.FixedProb(prob=epsilon),
    delay_step=delay_step,
    g_max=JI,
)
I2I = DeltaSynapse(
    I,
    I,
    bp.connect.FixedProb(prob=epsilon),
    delay_step=delay_step,
    g_max=JI,
)

# External population
ext = bp.dyn.PoissonGroup(
    E.num,  # So that the average number of connections to each population matches Ce
    nu,
    keep_size=False,
    sharding=None,
    spk_type=None,
    name=None,
    mode=None,
    seed=None,
)
ext2E = DeltaSynapse(
    ext,
    E,
    bp.connect.FixedProb(prob=epsilon),
    delay_step=delay_step,
    g_max=JE,
)
ext2I = DeltaSynapse(
    ext,
    I,
    bp.connect.FixedProb(prob=epsilon),
    delay_step=delay_step,
    g_max=JE,
)

# define input variables given to E/I populations
Ein = bp.dyn.InputVar(E.varshape)
Iin = bp.dyn.InputVar(I.varshape)
E.add_inp_fun("", Ein)
I.add_inp_fun("", Iin)



0
0
0
0
0
